# Pipeline DeeperHistReg: registro global no rígido y extracción de tiles pareados

Se implementa el pipeline de registro basado en DeeperHistReg sobre el mismo
par de fragmentos de tejido renal (T1, referencia; T2, móvil) utilizado en los
demás pipelines evaluados. Se mantienen sin cambios las etapas de detección de
tejido y de extracción de tiles bajo grilla común, variando únicamente el motor de registro empleado en la etapa intermedia.

## Configuración según la muestra a procesar

Este notebook procesa una muestra por ejecución. Para correrlo sobre Muestra1 o Muestra3, alcanza con modificar la ruta base del dataset a la carpeta correspondiente **en los dos bloques que la definen** -detección de tejido (GrandQC) y extracción de tiles-, antes de ejecutar el resto de las celdas:

```python
# Bloque GrandQC (detección de tejido)
ruta_dataset_imagenes = '/kaggle/input/datasets/candelapaez/muestra1nefro'   # o muestra3nefro

# Bloque de extracción de tiles
ruta_base_tif = '/kaggle/input/datasets/candelapaez/muestra1nefro'           # o muestra3nefro
```

Ambas variables deben apuntar siempre a la misma muestra. Los nombres de archivo (`muestra_1.tif`, `muestra_2.tif`) y el resto de la configuración no cambian entre muestras, ya que ambas siguen la misma convención T1 (referencia) / T2 (móvil).

## Detección de tejido con GrandQC

Se procede a la detección de tejido sobre cada fragmento (T1 y T2) de forma
independiente, mediante la arquitectura UNet++ con encoder EfficientNetB0 provista
por **GrandQC** (checkpoint `Tissue_Detection_MPP10.pth`, resolución de referencia
1,0 μm/px), que segmenta el tejido del fondo de vidrio y genera una máscara binaria
por archivo. La resolución real de cada imagen (MPP) se obtiene de sus metadatos
mediante `tiatoolbox`, empleando un valor de respaldo de 0,22 μm/px únicamente
cuando no puede leerse directamente del archivo.

In [ ]:
# DETECCION DE TEJIDO CON MODELO GRANDQC - TISSUE DETECTION
import os
import shutil
import re
import json

print("MODELO GRAND QC APLICADO A NEFROLOGIA")

print("\nVerificando Archivos en Kaggle...")

# ruta_dataset_imagenes cambiar segun la muestra
ruta_dataset_imagenes = '/kaggle/input/datasets/candelapaez/muestra3nefro'  
drive_td = '/kaggle/input/datasets/candelapaez/modelograndqc/Tissue_Detection_MPP10.pth'
drive_qc = '/kaggle/input/datasets/candelapaez/modelograndqc/GrandQC_MPP15.pth'

archivo_tif_t1 = 'muestra_1.tif'
archivo_tif_t2 = 'muestra_2.tif'
ruta_tif_t1 = os.path.join(ruta_dataset_imagenes, archivo_tif_t1)
ruta_tif_t2 = os.path.join(ruta_dataset_imagenes, archivo_tif_t2)

if os.path.exists(ruta_dataset_imagenes):
    archivos_tif = [f for f in os.listdir(ruta_dataset_imagenes) if f.endswith('.tif')]
    if archivos_tif:
        print(f"Éxito! Se encontraron {len(archivos_tif)} imagen(es): {archivos_tif}")
    else:
        print("La carpeta existe, pero no hay archivos .tif adentro.")
else:
    print(f"ERROR: No se encontró la carpeta {ruta_dataset_imagenes}")

if os.path.exists(drive_td) and os.path.exists(drive_qc):
    print("Éxito! Se encontraron los archivos .pth del modelo.")
else:
    print("ERROR: No se encontraron los modelos .pth. Revisa las rutas.")

print("\nInstalando tiatoolbox para leer el MPP real de cada archivo...")
!pip install -q tiatoolbox

from tiatoolbox.wsicore.wsireader import WSIReader

VALOR_MPP_FALLBACK = 0.22  

def leer_mpp_real(ruta_tif, fallback=VALOR_MPP_FALLBACK):
    try:
        lector = WSIReader.open(ruta_tif)
        mpp = float(lector.info.mpp[0])
        print(f"  [{os.path.basename(ruta_tif)}] MPP leído del metadato: {mpp:.4f} µm/px")
        return mpp
    except Exception as e:
        print(f"  [{os.path.basename(ruta_tif)}] No se pudo leer el MPP real ({e}). "
              f"Se usa el valor de emergencia: {fallback}")
        return fallback

mpp_t1 = leer_mpp_real(ruta_tif_t1)
mpp_t2 = leer_mpp_real(ruta_tif_t2)

carpeta_resultados = '/kaggle/working/Resultados_GrandQC'
os.makedirs(carpeta_resultados, exist_ok=True)
ruta_metadata_mpp = os.path.join(carpeta_resultados, "metadata_mpp.json")
with open(ruta_metadata_mpp, "w") as f:
    json.dump({archivo_tif_t1: mpp_t1, archivo_tif_t2: mpp_t2}, f, indent=2)
print(f"\nMPP guardado para comparación: {ruta_metadata_mpp}")

# Preparacion del Entorno 
%cd /kaggle/working
!rm -rf /kaggle/working/grandqc
!git clone https://github.com/cpath-ukk/grandqc.git /kaggle/working/grandqc

print("\n-> Instalando librerías requeridas...")
!apt-get update > /dev/null
!apt-get install -y openslide-tools > /dev/null
!pip install openslide-python segmentation_models_pytorch > /dev/null

base_path = '/kaggle/working/grandqc/01_WSI_inference_OPENSLIDE_QC/models'
os.makedirs(f"{base_path}/td", exist_ok=True)
os.makedirs(f"{base_path}/qc", exist_ok=True)

local_td = f"{base_path}/td/Tissue_Detection_MPP10.pth"
local_qc = f"{base_path}/qc/GrandQC_MPP15.pth"

try:
    shutil.copy(drive_td, local_td)
    shutil.copy(drive_qc, local_qc)
    print("\nModelos vinculados correctamente a la carpeta de trabajo.")
except FileNotFoundError:
    print(f"\nERROR: No se pudieron copiar los modelos .pth.")

sh_path = '/kaggle/working/grandqc/01_WSI_inference_OPENSLIDE_QC/run_tis.sh'
archivo_python = '/kaggle/working/grandqc/01_WSI_inference_OPENSLIDE_QC/wsi_tis_detect.py'

with open(sh_path, 'r') as file:
    sh_content_original = file.read()
with open(archivo_python, 'r') as file:
    codigo_original = file.read()

tejidos_a_procesar = [
    (ruta_tif_t1, mpp_t1, '/kaggle/working/slide_temp_t1'),
    (ruta_tif_t2, mpp_t2, '/kaggle/working/slide_temp_t2'),
]

for ruta_tif_actual, mpp_actual, carpeta_temp in tejidos_a_procesar:
    os.makedirs(carpeta_temp, exist_ok=True)
    ruta_temp = os.path.join(carpeta_temp, os.path.basename(ruta_tif_actual))
    if not os.path.exists(ruta_temp):
        try:
            os.symlink(ruta_tif_actual, ruta_temp)
        except OSError:
            shutil.copy(ruta_tif_actual, ruta_temp)

    sh_mod = re.sub(r"^SLIDE_FOLDER=.*", f"SLIDE_FOLDER='{carpeta_temp}'", sh_content_original, flags=re.MULTILINE)
    sh_mod = re.sub(r"^OUTPUT_DIR=.*", f"OUTPUT_DIR='{carpeta_resultados}'", sh_mod, flags=re.MULTILINE)
    with open(sh_path, 'w') as f:
        f.write(sh_mod)

    # Parchear wsi_tis_detect.py con el MPP REAL de ESTE archivo
    patron_busqueda = r"slide\.properties\[['\"]openslide\.mpp-x['\"]\]"
    reemplazo = f"slide.properties.get('openslide.mpp-x', {mpp_actual})"
    codigo_mod = re.sub(patron_busqueda, reemplazo, codigo_original)
    silenciador = "import warnings\nwarnings.filterwarnings('ignore')\n"
    codigo_mod = silenciador + codigo_mod
    with open(archivo_python, 'w') as f:
        f.write(codigo_mod)

    print(f"\nCorriendo GrandQC tissue detection sobre {os.path.basename(ruta_tif_actual)} "
          f"(mpp={mpp_actual:.4f})...")
    %cd /kaggle/working/grandqc/01_WSI_inference_OPENSLIDE_QC
    !sh run_tis.sh

print("\nGrandQC finalizado para ambos tejidos.")
print(f"Máscaras generadas en: {carpeta_resultados}/tis_det_mask")

## Definición del recorte de tejido para DeeperHistReg

A partir de las máscaras de GrandQC, se calcula el bounding box de tejido de cada
fragmento, fusionando sus contornos válidos en un único rectángulo. El bounding box se proyecta a resolución completa
(nivel 0) y se recorta con un margen fijo de 300 px por lado, directamente sobre
el archivo original y sin reescalado. Los recortes de T1 y T2, aún sin registrar
entre sí, se exportan como TIF piramidal mediante `pyvips`, formato de entrada
requerido por DeeperHistReg.

In [ ]:
# DEFINICION DEL RECORTE PARA DEEPERHISTREG
!apt-get -qq update
!apt-get -qq install -y libvips > /dev/null

!pip install -q deeperhistreg pyvips

import torch
print("GPU disponible:", torch.cuda.is_available())
print("Device:", "cuda:0" if torch.cuda.is_available() else "cpu")

import cv2
import numpy as np
import matplotlib.pyplot as plt
import openslide
import os
import json
import itertools
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

try:
    import SimpleITK as sitk
    SITK_DISPONIBLE = True
except ImportError:
    SITK_DISPONIBLE = False
    print("AVISO: SimpleITK no está instalado (pip install SimpleITK).")

# ruta_base_tif CAMBIAR según la muestra
ruta_base_tif = '/kaggle/input/datasets/candelapaez/muestra3nefro'  
ruta_base_mascaras = '/kaggle/working/Resultados_GrandQC/tis_det_mask'

archivo_tif_t1 = 'muestra_1.tif'   
archivo_tif_t2 = 'muestra_2.tif'   

ruta_tif_t1 = os.path.join(ruta_base_tif, archivo_tif_t1)
ruta_tif_t2 = os.path.join(ruta_base_tif, archivo_tif_t2)

ruta_mascara_t1 = os.path.join(ruta_base_mascaras, archivo_tif_t1 + '_MASK.png')
ruta_mascara_t2 = os.path.join(ruta_base_mascaras, archivo_tif_t2 + '_MASK.png')

ruta_offsets_json = os.path.join(ruta_base_tif, 'offsets_recorte.json')

carpeta_base = '/kaggle/working/tiles_apareados_deeperhistreg'
carpeta_ecc_t1 = f'{carpeta_base}/ecc/tejido1'
carpeta_ecc_t2 = f'{carpeta_base}/ecc/tejido2'
carpeta_geom_t1 = f'{carpeta_base}/geom/tejido1'
carpeta_geom_t2 = f'{carpeta_base}/geom/tejido2'
for carpeta in [carpeta_ecc_t1, carpeta_ecc_t2, carpeta_geom_t1, carpeta_geom_t2]:
    os.makedirs(carpeta, exist_ok=True)

carpeta_dhr = '/kaggle/working/deeperhistreg_work'
os.makedirs(carpeta_dhr, exist_ok=True)

# Parámetros 
TILE_SIZE = 512
STRIDE = 256
MIN_TEJIDO_PCT = 0.40
MIN_INFORMATIVIDAD = 15.0
PAD = 128
WORK_SIZE = TILE_SIZE + 2 * PAD
MARGEN_RECORTE_TEJIDO = 300  

print(f"TILE_SIZE={TILE_SIZE}  STRIDE={STRIDE}  MIN_TEJIDO_PCT={MIN_TEJIDO_PCT}  "
      f"MIN_INFORMATIVIDAD={MIN_INFORMATIVIDAD}  PAD={PAD}")


def mascara_tejido_rgb(img_rgb):
    gray = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY)
    _, m = cv2.threshold(gray, 240, 255, cv2.THRESH_BINARY_INV)
    k = np.ones((5, 5), np.uint8)
    return cv2.morphologyEx(m, cv2.MORPH_CLOSE, k, iterations=2)


def medir_informatividad(img_rgb, mask=None):
    gray = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY)
    if mask is not None and np.sum(mask > 0) > 0:
        valores = gray[mask > 0]
        if valores.size < 100:
            return 0.0
        lap = cv2.Laplacian(gray, cv2.CV_64F)
        return float(lap[mask > 0].var())
    lap = cv2.Laplacian(gray, cv2.CV_64F)
    return float(lap.var())


_clahe_confianza = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))

def medir_confianza_alineacion(img1_rgb, img2_rgb):
    g1 = cv2.cvtColor(img1_rgb, cv2.COLOR_RGB2GRAY)
    g2 = cv2.cvtColor(img2_rgb, cv2.COLOR_RGB2GRAY)
    g1 = _clahe_confianza.apply(g1)
    g2 = _clahe_confianza.apply(g2)

    gx1 = cv2.Sobel(g1, cv2.CV_32F, 1, 0, ksize=3)
    gy1 = cv2.Sobel(g1, cv2.CV_32F, 0, 1, ksize=3)
    f1 = cv2.magnitude(gx1, gy1)

    gx2 = cv2.Sobel(g2, cv2.CV_32F, 1, 0, ksize=3)
    gy2 = cv2.Sobel(g2, cv2.CV_32F, 0, 1, ksize=3)
    f2 = cv2.magnitude(gx2, gy2)

    win = cv2.createHanningWindow((f1.shape[1], f1.shape[0]), cv2.CV_32F)
    (dx, dy), response = cv2.phaseCorrelate(f1, f2, win)
    return {
        "shift_residual_px": float(np.hypot(dx, dy)),
        "confianza_alineacion": float(response),
    }


def chequeo_ecc_diagnostico(img1_rgb, img2_rgb):
    gray1 = cv2.cvtColor(img1_rgb, cv2.COLOR_RGB2GRAY)
    gray2 = cv2.cvtColor(img2_rgb, cv2.COLOR_RGB2GRAY)

    warp_mode = cv2.MOTION_EUCLIDEAN
    warp_matrix = np.eye(2, 3, dtype=np.float32)
    iteraciones = 50
    tolerancia = 1e-4
    criteria = (cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, iteraciones, tolerancia)

    try:
        cv2.findTransformECC(gray1, gray2, warp_matrix, warp_mode, criteria)
        return True
    except Exception:
        return False


def bbox_global(contornos):
    rects = [cv2.boundingRect(c) for c in contornos]
    xn = min(x for x, y, w, h in rects)
    yn = min(y for x, y, w, h in rects)
    xx = max(x + w for x, y, w, h in rects)
    yx = max(y + h for x, y, w, h in rects)
    return xn, yn, xx - xn, yx - yn


def obtener_bbox_tejido(slide_path, mask_path):
    """Bbox (Nivel 0) del tejido de UN tif individual, a partir de su propia
    máscara GrandQC."""
    slide = openslide.OpenSlide(slide_path)
    try:
        macro_level = min(3, slide.level_count - 1)
        macro_dims = slide.level_dimensions[macro_level]
        ds = slide.level_downsamples[macro_level]

        qc_mask = np.array(Image.open(mask_path).convert('L'), dtype=np.uint8)
        if qc_mask.max() <= 1:
            qc_mask = (qc_mask * 255).astype(np.uint8)
        _, binary_mask_full = cv2.threshold(qc_mask, 127, 255, cv2.THRESH_BINARY)
        if binary_mask_full[0, 0] == 255:
            binary_mask_full = cv2.bitwise_not(binary_mask_full)

        k_close = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (15, 15))
        binary_mask_full = cv2.morphologyEx(binary_mask_full, cv2.MORPH_CLOSE, k_close)
        binary_mask_full = cv2.morphologyEx(binary_mask_full, cv2.MORPH_DILATE, k_close)

        w_macro, h_macro = macro_dims
        binary_mask_macro = np.array(
            Image.fromarray(binary_mask_full).resize((w_macro, h_macro), Image.NEAREST)
        )

        contours, _ = cv2.findContours(binary_mask_macro, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        valid_contours = [c for c in contours if cv2.contourArea(c) > 200]
        if not valid_contours:
            raise RuntimeError(f"No se detectó tejido en la máscara: {mask_path}")

        xm, ym, wm, hm = bbox_global(valid_contours)
        x0, y0, w0, h0 = int(xm * ds), int(ym * ds), int(wm * ds), int(hm * ds)
        return (x0, y0, w0, h0), slide.level_dimensions[0]
    finally:
        slide.close()


print("Detectando bbox de tejido en muestra_1.tif (tejido1 / referencia)...")
tejido1_bbox, dims_nivel0_t1 = obtener_bbox_tejido(ruta_tif_t1, ruta_mascara_t1)
print("  bbox_nivel0:", tejido1_bbox, " dims:", dims_nivel0_t1)

print("Detectando bbox de tejido en muestra_2.tif (tejido2 / móvil)...")
tejido2_bbox, dims_nivel0_t2 = obtener_bbox_tejido(ruta_tif_t2, ruta_mascara_t2)
print("  bbox_nivel0:", tejido2_bbox, " dims:", dims_nivel0_t2)

with open(ruta_offsets_json, 'r') as f:
    offsets_recorte = json.load(f)

print("Contenido de offsets_recorte.json:", offsets_recorte)

offset_t1 = offsets_recorte.get(archivo_tif_t1, offsets_recorte.get('muestra_1', {'x': 0, 'y': 0}))
offset_t2 = offsets_recorte.get(archivo_tif_t2, offsets_recorte.get('muestra_2', {'x': 0, 'y': 0}))

import pyvips

def recortar_tejido_nivel0(slide_path, bbox_nivel0, margen, dims_nivel0):
    x0, y0, w0, h0 = bbox_nivel0
    x0m = max(0, x0 - margen)
    y0m = max(0, y0 - margen)
    x1m = min(dims_nivel0[0], x0 + w0 + margen)
    y1m = min(dims_nivel0[1], y0 + h0 + margen)
    w0m, h0m = x1m - x0m, y1m - y0m

    slide = openslide.OpenSlide(slide_path)
    try:
        region_rgba = slide.read_region((x0m, y0m), 0, (w0m, h0m))
        region_rgb = np.array(region_rgba.convert('RGB'))
    finally:
        slide.close()

    origen_nivel0 = (x0m, y0m)  
    return region_rgb, origen_nivel0


print("Recortando Tejido 1 (referencia) a resolución completa...")
t1_rgb_full, origen_t1_local = recortar_tejido_nivel0(ruta_tif_t1, tejido1_bbox, MARGEN_RECORTE_TEJIDO, dims_nivel0_t1)
print(f"  Tejido 1: shape={t1_rgb_full.shape}  origen_local(dentro de muestra_1.tif)={origen_t1_local}")

print("Recortando Tejido 2 (móvil) a resolución completa...")
t2_rgb_full, origen_t2_local = recortar_tejido_nivel0(ruta_tif_t2, tejido2_bbox, MARGEN_RECORTE_TEJIDO, dims_nivel0_t2)
print(f"  Tejido 2: shape={t2_rgb_full.shape}  origen_local(dentro de muestra_2.tif)={origen_t2_local}")


origen_t1 = (origen_t1_local[0] + offset_t1.get('x', 0), origen_t1_local[1] + offset_t1.get('y', 0))
origen_t2 = (origen_t2_local[0] + offset_t2.get('x', 0), origen_t2_local[1] + offset_t2.get('y', 0))

ruta_target_tif = f"{carpeta_dhr}/tejido1_target.tif"   
ruta_source_tif = f"{carpeta_dhr}/tejido2_source.tif"  

pyvips.Image.new_from_array(t1_rgb_full).tiffsave(ruta_target_tif, tile=True, compression="lzw")
pyvips.Image.new_from_array(t2_rgb_full).tiffsave(ruta_source_tif, tile=True, compression="lzw")
print("TIFs exportados para DeeperHistReg.")

## Registro con DeeperHistReg, extracción de tiles y visualización

Se ejecuta el registro entre T1 y T2 mediante DeeperHistReg, con la configuración
predefinida `default_nonrigid_high_resolution` (etapa rígida por descriptores
profundos seguida de una etapa no rígida de optimización de instancia), y se
verifica que el `pad_value` interno de la librería sea consistente con el fondo
blanco (255) del resto del pipeline. Se configura `copy_target=False`, de modo
que T1 no sea reescrito por la librería: el tile de T1 usado en adelante proviene
directamente del recorte original, sin interpolación adicional. El único tejido
efectivamente deformado es T2, cuyas dimensiones se verifican y, de ser
necesario, se re-escalan para calzar exactamente con T1.

Sobre el par ya alineado se aplica la misma grilla de extracción de los demás
pipelines (tile de 512 px, stride de 256 px), sin transformaciones ni
deformaciones adicionales por tile, dado que el registro no rígido ya se aplicó
una única vez sobre la imagen completa. Se conservan los mismos filtros de
tejido e informatividad, y la misma clasificación diagnóstica "ecc"/"geom" según
la convergencia del chequeo ECC, sin que esta afecte el resultado del registro.
Por último, se visualiza una muestra aleatoria de tiles pareados, con la
superposición de T1 y T2 y su ubicación sobre el recorte de referencia.

In [ ]:
# REGISTRO CON DEEPERHISTREG 
import deeperhistreg
import torch
from tqdm.auto import tqdm

USAR_PRESET_RAPIDO = False  

if USAR_PRESET_RAPIDO:
    registration_parameters = deeperhistreg.configs.default_nonrigid_fast()
else:
    registration_parameters = deeperhistreg.configs.default_nonrigid_high_resolution()

registration_parameters['device'] = "cuda:0" if torch.cuda.is_available() else "cpu"
registration_parameters['nonrigid_registration_params']['device'] = registration_parameters['device']


print("pad_value (fondo) configurado en DeeperHistReg:",
      registration_parameters['loading_params']['pad_value'])

config = {
    'source_path': ruta_source_tif,          
    'target_path': ruta_target_tif,          
    'output_path': f"{carpeta_dhr}/output",
    'registration_parameters': registration_parameters,
    'case_name': "nefro_dhr_registro",
    'save_displacement_field': True,         
    'copy_target': False,  
    'delete_temporary_results': True,       
    'temporary_path': f"{carpeta_dhr}/tmp",
}

print("\nCorriendo registro DeeperHistReg (rígido con features profundas + no-rígido)...")
deeperhistreg.run_registration(**config)
print("Registro finalizado. Salida en:", config['output_path'])


archivos_salida = os.listdir(config['output_path'])
print("Archivos generados:", archivos_salida)

ruta_warped = [os.path.join(config['output_path'], f) for f in archivos_salida if "warped_source" in f][0]


t2_registrado = pyvips.Image.tiffload(ruta_warped).numpy()

print("Shape Tejido 1 (t1_rgb_full, directo del recorte original):", t1_rgb_full.shape)
print("Shape Tejido 2 registrado (warped_source):", t2_registrado.shape)

if t1_rgb_full.shape[:2] != t2_registrado.shape[:2]:
    print("\nATENCIÓN: las dimensiones no coinciden exactamente. "
          "Se re-escala SOLO T2 (registrado) al tamaño de t1_rgb_full antes de "
          "tilear (esto NO debería pasar con final_level=0, pero se deja el "
          "chequeo por seguridad). T1 nunca se toca.")
    t2_registrado = cv2.resize(t2_registrado, (t1_rgb_full.shape[1], t1_rgb_full.shape[0]),
                                interpolation=cv2.INTER_LINEAR)
else:
    print("\nDimensiones coinciden. Se procede a extraer los tiles ...")

t1_rgb_reg = t1_rgb_full  
t2_rgb_reg = t2_registrado[:, :, :3] if t2_registrado.shape[-1] > 3 else t2_registrado


# EXTRACCIÓN DE TILES
h_reg, w_reg = t1_rgb_reg.shape[:2]

xs = range(0, w_reg - TILE_SIZE, STRIDE)
ys = range(0, h_reg - TILE_SIZE, STRIDE)
total_candidatos = len(xs) * len(ys)

tiles_ecc = 0
tiles_geom = 0
tiles_descartados = 0
tiles_descartados_tejido = 0
tiles_descartados_informatividad = 0
pares_validos = []

print(f"Analizando grilla: {len(xs)} columnas x {len(ys)} filas ({total_candidatos} candidatos)...")
print(f"Filtro de tejido mínimo: {MIN_TEJIDO_PCT*100:.0f}%")
print(f"Filtro de informatividad (varianza Laplaciano): >= {MIN_INFORMATIVIDAD}")

pbar = tqdm(itertools.product(ys, xs), total=total_candidatos, desc="Extrayendo tiles", unit="tile")

for y1, x1 in pbar:
    t1_tile = t1_rgb_reg[y1:y1+TILE_SIZE, x1:x1+TILE_SIZE]
    t2_tile = t2_rgb_reg[y1:y1+TILE_SIZE, x1:x1+TILE_SIZE]

    if t1_tile.shape[:2] != (TILE_SIZE, TILE_SIZE) or t2_tile.shape[:2] != (TILE_SIZE, TILE_SIZE):
        tiles_descartados += 1
        continue

    # Filtro de tejido
    mask_t1 = mascara_tejido_rgb(t1_tile)
    mask_t2 = mascara_tejido_rgb(t2_tile)
    pct_t1 = np.sum(mask_t1 > 0) / (TILE_SIZE ** 2)
    pct_t2 = np.sum(mask_t2 > 0) / (TILE_SIZE ** 2)
    if pct_t1 < MIN_TEJIDO_PCT or pct_t2 < MIN_TEJIDO_PCT:
        tiles_descartados += 1
        tiles_descartados_tejido += 1
        pbar.set_postfix(guardados=tiles_ecc + tiles_geom, ecc=tiles_ecc, geom=tiles_geom,
                          descartados=tiles_descartados)
        continue

    # Filtro de informatividad 
    info_t1 = medir_informatividad(t1_tile, mask_t1)
    info_t2 = medir_informatividad(t2_tile, mask_t2)
    if info_t1 < MIN_INFORMATIVIDAD or info_t2 < MIN_INFORMATIVIDAD:
        tiles_descartados += 1
        tiles_descartados_informatividad += 1
        pbar.set_postfix(guardados=tiles_ecc + tiles_geom, ecc=tiles_ecc, geom=tiles_geom,
                          descartados=tiles_descartados)
        continue

    # Métrica de confianza de alineación
    confianza = medir_confianza_alineacion(t1_tile, t2_tile)

    # Chequeo ECC diagnóstico (solo para clasificar ecc/ vs geom/, no deforma) 
    ecc_ok = chequeo_ecc_diagnostico(t1_tile, t2_tile)

    # Coordenadas absolutas de Nivel 0 
    x1_orig = origen_t1[0] + x1
    y1_orig = origen_t1[1] + y1

    # Coordenadas LOCALES dentro de muestra_1.tif 
    x1_map = origen_t1_local[0] + x1
    y1_map = origen_t1_local[1] + y1

    idx = tiles_ecc + tiles_geom
    nombre = f"tile_{idx:05d}_r{y1}_c{x1}.png"   

    if ecc_ok:
        Image.fromarray(t1_tile).save(f"{carpeta_ecc_t1}/{nombre}")
        Image.fromarray(t2_tile).save(f"{carpeta_ecc_t2}/{nombre}")
        tiles_ecc += 1
        carpeta_tag = "ecc"
    else:
        Image.fromarray(t1_tile).save(f"{carpeta_geom_t1}/{nombre}")
        Image.fromarray(t2_tile).save(f"{carpeta_geom_t2}/{nombre}")
        tiles_geom += 1
        carpeta_tag = "geom"

    pares_validos.append({
        'nombre': nombre,
        'carpeta': carpeta_tag,
        'x1': x1, 'y1': y1,
        'x1_orig': x1_orig, 'y1_orig': y1_orig,  
        'x1_map': x1_map, 'y1_map': y1_map,       
        'ecc_exitoso': ecc_ok,
        'informatividad_t1': round(info_t1, 2),
        'informatividad_t2': round(info_t2, 2),
        'shift_residual_px': round(confianza['shift_residual_px'], 2),
        'confianza_alineacion': round(confianza['confianza_alineacion'], 4),
    })

    pbar.set_postfix(guardados=tiles_ecc + tiles_geom, ecc=tiles_ecc, geom=tiles_geom,
                      descartados=tiles_descartados)

pbar.close()

total_guardados = tiles_ecc + tiles_geom
total_analizados = total_guardados + tiles_descartados
pct_exito = (total_guardados / total_analizados * 100) if total_analizados > 0 else 0

print(f"\nEXTRACCIÓN FINALIZADA (DeeperHistReg)")
print(f"Resumen ({total_analizados} recuadros analizados):")
print(f" -> ecc/  (ECC converge sobre el resultado de DeeperHistReg): {tiles_ecc} tiles")
print(f" -> geom/ (ECC no converge, solo registro de DeeperHistReg):  {tiles_geom} tiles")
print(f" -> Total guardados: {total_guardados} ({pct_exito:.1f}%)")
print(f" -> Descartados por tejido insuficiente (<{MIN_TEJIDO_PCT*100:.0f}%): {tiles_descartados_tejido}")
print(f" -> Descartados por baja informatividad (<{MIN_INFORMATIVIDAD}): {tiles_descartados_informatividad}")


manifest_path = f"{carpeta_base}/indice_tiles.csv"
df_manifest = pd.DataFrame(pares_validos)
df_manifest['ruta_t1'] = df_manifest.apply(
    lambda r: f"{carpeta_ecc_t1 if r['ecc_exitoso'] else carpeta_geom_t1}/{r['nombre']}", axis=1)
df_manifest['ruta_t2'] = df_manifest.apply(
    lambda r: f"{carpeta_ecc_t2 if r['ecc_exitoso'] else carpeta_geom_t2}/{r['nombre']}", axis=1)
df_manifest.to_csv(manifest_path, index=False)
print(f"\nÍndice guardado en: {manifest_path}  (columnas x1_orig/y1_orig incluidas)")

if len(df_manifest) > 0:
    print(f"\nConfianza de alineación (mediana): {df_manifest['confianza_alineacion'].median():.3f}")


# VISUALIZACIÓN DE RESULTADOS 
import random

def visualizar_resultados_dhr(df_manifest, n_muestras=6):
    if len(df_manifest) == 0:
        print("No hay pares para mostrar.")
        return

    muestra = df_manifest.sample(min(n_muestras, len(df_manifest))).to_dict('records')


    fig, axes = plt.subplots(len(muestra), 3, figsize=(15, 5 * len(muestra)))
    if len(muestra) == 1:
        axes = [axes]

    for i, par in enumerate(muestra):
        tipo = "ECC" if par['ecc_exitoso'] else "GEOM"
        t1 = np.array(Image.open(par['ruta_t1']))
        t2 = np.array(Image.open(par['ruta_t2']))
        superpuesto = cv2.addWeighted(t1, 0.5, t2, 0.5, 0)
    
        axes[i][0].imshow(t1)
        axes[i][0].set_title(f"T1 (Ref)\n{par['nombre']}\ninfo={par['informatividad_t1']:.0f}")
        axes[i][0].axis('off')

        axes[i][1].imshow(t2)
        axes[i][1].set_title(f"T2 ({tipo})\ninfo={par['informatividad_t2']:.0f}")
        axes[i][1].axis('off')

        axes[i][2].imshow(superpuesto)
        axes[i][2].set_title(f"Superposición\nconfianza={par['confianza_alineacion']:.3f}")
        axes[i][2].axis('off')

    plt.tight_layout()
    plt.show()


print("\nVISUALIZACIÓN DE RESULTADOS (DeeperHistReg)")
cantidad_mostrar = 10
print(f"Generando visualización de {cantidad_mostrar} muestras aleatorias...")
visualizar_resultados_dhr(df_manifest, n_muestras=cantidad_mostrar) 

## Exportación del conjunto de tiles pareados

Finalmente, se comprime el conjunto completo de tiles pareados generado -incluyendo
los subconjuntos clasificados como "ecc" y "geom" para ambos tejidos- en un archivo
`.zip`, quedando disponible para su descarga y su posterior incorporación al
cálculo de métricas de calidad de alineación descripto en la sección de Métricas.

La carpeta resultante se organiza según el resultado del chequeo diagnóstico de
convergencia ECC y el tejido correspondiente. A diferencia de Sift+Demons, esta
clasificación es puramente informativa: en ambos casos el tile de T2 proviene del
mismo registro de DeeperHistReg, sin que la convergencia de ECC modifique el
resultado ni implique la aplicación de un ajuste adicional.

```
tiles_apareados_deeperhistreg/
├── ecc/                    ← tiles donde el chequeo ECC converge sobre el
│   ├── tejido1/               resultado de DeeperHistReg (clasificación diagnóstica)
│   └── tejido2/
└── geom/                   ← tiles donde el chequeo ECC no converge
    ├── tejido1/               (mismo registro de DeeperHistReg, sin ajuste adicional)
    └── tejido2/
```

In [ ]:
# GUARDADO DE TILES EN CARPETA .ZIP
import shutil
import os
from IPython.display import FileLink, display


# Compresión y exportación del dataset
os.chdir('/kaggle/working')

carpeta_tiles = '/kaggle/working/tiles_apareados_deeperhistreg'
nombre_zip_base = '/kaggle/working/TilesDeeperhistregMuestra3'

print(f"\nComprimiendo la carpeta: {carpeta_tiles}...")

# Comprimir la carpeta entera en formato ZIP
shutil.make_archive(nombre_zip_base, 'zip', carpeta_tiles)

archivo_generado = f"{nombre_zip_base}.zip"
peso_mb = os.path.getsize(archivo_generado) / (1024 * 1024)


print(f"\nCompresión finalizada!")
print(f"Archivo: {archivo_generado}")
print(f"Peso aproximado: {peso_mb:.2f} MB")

# Generar un link para descargar 
display(FileLink('TilesDeeperhistregMuestra3.zip'))